In [43]:
from math import pi
import numpy as np
import matplotlib.pyplot as plt
import time

#import pyvista as pv
from time import strftime
import importlib


from crystal import *
from detector import *
from beam import *
from ptycho_fs import *
from utils import *
from functions import *

from sequence import simulate_one_bragg_order

import detector
import beam 
import functions
save_path = '/home/mohahmed/CBD_sim/Results/'

In [44]:
import multiprocessing

multiprocessing.cpu_count()


96

In [45]:
importlib.reload(detector)
importlib.reload(beam)
importlib.reload(functions)

<module 'functions' from '/home/mohahmed/pycbd/functions.py'>

# Set up

In [46]:
#Initialise Crystal
crystal = Crystal()
crystal.unit_cell_shape = (90,90,90)
crystal.unit_cell_size = (4.025, 4.025, 4.025)
crystal.crystal_size = (50,50,6)
crystal.max_hkl = 1

crystal.set_real_lattice_vectors()
crystal.crystal_orientation = (45,0,18,0)
crystal.rotate_UC()

atoms = {'Au': [0,0,0]}
cube_normals = cuboid_normals(crystal.crystal_size)
padding = (5,5,5) # Zero padding outside crystal

crystal.set_recip_lattice_vectors()
crystal.set_shape_array(cube_normals, padding=padding)

for key, value in atoms.items():
    crystal.add_atom(key, value)

In [47]:
# Initialise Beam
beam = Beam()
beam.energy = 17
beam.lens_focal_length = 0
beam.focusing_lens_NA = 0.02
beam.set_convergent_kins(int(1e7))

#Optional
beam.amplitude_profile = {
   'gaussian_sigma' : 0,
   'absorption_coeff': 0,
   'tophat_radius': 10,
}

beam.lens_aberrations = {
    'defocus': 0,
    'spherical':0,
    'coma':0,
    'astigmatism':0,
}

beam.focus = beam.wavelength/beam.focusing_lens_NA * 5
beam_focus = (beam.focus // crystal.unit_cell_size[:2]).astype(int)
stride = beam_focus + 1
ill_vol = ptycho_scan_volumes(crystal.crystal_size, stride=stride, beam_focus=beam_focus, padding=padding)
print(f"beam_focus is {beam_focus}")
print(f"The illumination volumes are: {ill_vol}")

beam_focus is [45 45]
The illumination volumes are: [(0, 45, 0, 45, 0, 6)]


In [48]:
# Initialise Detector
detector = Detector()
detector.distance = 0.16
detector.size = (0.12,0.12)
detector.pixel_size = (75,75)


In [88]:
crystal.rlvs.shape

(2097152, 3)

# Sim

In [90]:
# Simulation Parameters
range_scale = .45
grid_points = 256
hkl = (1,1,-1)
intensity_filter = 0.01
elasticity_tolerance = 1e-5

# Generate reciprocal lattice point for one hkl
crystal.gen_RLS_one_hkl(grid_points = grid_points, 
                        hkl=hkl, 
                        range_scale = range_scale, 
                        ravel=True)

# Compute the G vector for the chosen HKL
hkl = np.array(hkl)
crystal.gvectors = np.dot(hkl, crystal._recip_space_lattice)[None,:]

# # Compute reciprocal lattice intensity and 
# reciprocal_lattice_intensity = np.abs(calculate_scattering_amplitude(crystal.real_space_vectors, 
#                                                               crystal.rlvs, 
#                                                               crystal.real_space_coords, 
#                                                               crystal.atoms_list, 
#                                                               list(crystal.loc_atoms.values()), 
#                                                               beam.wavelength, 
#                                                               mask_Ri=None,
#                                                               n_jobs = 80))**2

# # Filter the RLVS according to intensity
# absolute_intensity_filter = intensity_filter * reciprocal_lattice_intensity.max()
# crystal.rlvs = crystal.rlvs[reciprocal_lattice_intensity>absolute_intensity_filter]

# Compute the kout vectors from reciprocal lattice vectors and kin
kouts, kin_indices, _ = compute_kout_from_G_kin(crystal.gvectors, beam.kins)
kins = beam.kins[kin_indices]  

# Filter the kouts and kins to satisfy only elastic scattering condition
kouts, kins_opt, mask = filter_elastic_scatt(kouts, 
                                             kins, 
                                             tolerance=elasticity_tolerance, 
                                             wavelength= beam.wavelength)    

# Compute scattering vectors in the experiment
qvecs = kouts-kins_opt

# Index Q 
q_indexed = indexQ(qvecs, crystal._recip_space_lattice)
q_indices = np.unique(q_indexed, axis=0)

# Set up image detector
nx,ny = (np.array(detector.size)/(np.array(detector.pixel_size)*1e-6)).astype(int)
full_image = np.zeros((nx,ny), dtype=complex)


In [91]:
q_indices

array([[ 1,  1, -1]])

In [94]:
scat_amp_g = calculate_scattering_amplitude_chunked_v2(
                crystal.real_space_vectors,
                crystal.rlvs,
                crystal.real_space_coords,
                crystal.atoms_list,
                list(crystal.loc_atoms.values()),
                beam.wavelength,
                mask_Ri=None,
                n_jobs=80,
                q_batch_size=10000
            )

scattering chunks: 100%|██████████| 1678/1678 [48:29<00:00,  1.73s/it]


In [95]:
scat_amp_g.shape

(16777216,)

In [96]:
66*64

4224

In [ ]:
# Set up image detector
nx,ny = (np.array(detector.size)/(np.array(detector.pixel_size)*1e-6)).astype(int)
full_image = np.zeros((nx,ny))

# Pupil function 
pupil_func = 1.0 * np.exp(1j*beam.lens_aberrations)
kin_pupil_func = pupil_func[mask]
pupil_image = generate_detector_image(np.abs(pupil_func), kins, detector.size, detector.pixel_size, detector.distance)
full_image += pupil_image

shift = np.array([[1,1,0]])
phase_shift = compute_phase_shift(crystal.rlvs, shift, n_jobs = 50, batch_size=1000)

shifted_scatter = phase_shift.flatten() * scat_amp_g


# for k, a_k in tqdm(zip(kins_opt, kin_pupil_func)):
        
#     kout = crystal.rlvs + k 
#     scattered =  a_k * shifted_scatter
    
#     image = generate_detector_image(scattered, kout, detector.size, detector.pixel_size, detector.distance)
#     full_image += image


rge/envs/pyxtools/lib/python3.10/multiprocessing/queues.py", line 108, in get
    if not self._rlock.acquire(block, timeout):
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/gpfs/cfel/user/mohahmed/miniforge/envs/pyxtools/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/gpfs/cfel/user/mohahmed/miniforge/envs/pyxtools/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/gpfs/cfel/user/mohahmed/miniforge/envs/pyxtools/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py", line 438, in _process_worker
    previous_tb = traceback.format_exc()
  File "/gpfs/cfel/user/mohahmed/miniforge/envs/pyxtools/lib/python3.10/traceback.py", line 183, in format_exc
    return ""

In [ ]:
def process_one(k, a_k):
    kout = crystal.rlvs + k
    scattered = a_k * shifted_scatter
    
    return generate_detector_image(scattered, kout,
                                   detector.size, detector.pixel_size, detector.distance)

images = Parallel(n_jobs=-1)(delayed(process_one)(k, a_k)
                             for k, a_k in zip(kins_opt, kin_pupil_func))

full_image += sum(images)

, value, tb, limit=limit, compact=True)
  File "/gpfs/cfel/user/mohahmed/miniforge/envs/pyxtools/lib/python3.10/traceback.py", line 502, in __init__
    self.stack = StackSummary.extract(
KeyboardInterrupt
  File "/gpfs/cfel/user/mohahmed/miniforge/envs/pyxtools/lib/python3.10/traceback.py", line 383, in extract
    f.line
  File "/gpfs/cfel/user/mohahmed/miniforge/envs/pyxtools/lib/python3.10/traceback.py", line 306, in line
    self._line = linecache.getline(self.filename, self.lineno)
  File "/gpfs/cfel/user/mohahmed/miniforge/envs/pyxtools/lib/python3.10/linecache.py", line 30, in getline
    lines = getlines(filename, module_globals)
  File "/gpfs/cfel/user/mohahmed/miniforge/envs/pyxtools/lib/python3.10/linecache.py", line 46, in getlines
    return updatecache(filename, module_globals)
  File "/gpfs/cfel/user/mohahmed/miniforge/envs/pyxtools/lib/python3.10/linecache.py", line 93, in updatecache
    stat = os.stat(fullname)
KeyboardInterrupt
Traceback (most recent call last):
  F

In [ ]:
%matplotlib ipympl
plt.figure()
plt.imshow(np.log(np.abs(full_image)+1e-8),vmin=2,vmax=50)

plt.show()

ile "/gpfs/cfel/user/mohahmed/miniforge/envs/pyxtools/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py", line 426, in _process_worker
    call_item = call_queue.get(block=True, timeout=timeout)
  File "/gpfs/cfel/user/mohahmed/miniforge/envs/pyxtools/lib/python3.10/multiprocessing/queues.py", line 108, in get
    if not self._rlock.acquire(block, timeout):
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/gpfs/cfel/user/mohahmed/miniforge/envs/pyxtools/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/gpfs/cfel/user/mohahmed/miniforge/envs/pyxtools/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/gpfs/cfel/user/mohahmed/miniforge/envs/pyxtools/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py", line 438, in _process_worker
    previous_tb = traceback.forma